In [ ]:
import pandas as pd
import numpy as np

# Veriyi yükleme
df = pd.read_csv('../data_generation/output/transactions.csv')
df['created_at'] = pd.to_datetime(df['created_at'])

# Üst Yönetici Özeti (Özet KPI Rakamları)
toplam_islem = len(df)
toplam_hacim = df['amount'].sum()

# Sütunların varlığını kontrol ederek anomali sayısını güvenli hesaplama
if 'is_anomaly' in df.columns:
    anomali_sayisi = df[df['is_anomaly'] == 1].shape[0]
elif 'anomaly_type' in df.columns:
    anomali_sayisi = df[df['anomaly_type'] != 'Normal'].shape[0]
else:
    anomali_sayisi = 0 # Eğer CSV'de henüz etiketleme yoksa hata vermemesi için

anomali_orani = (anomali_sayisi / toplam_islem) * 100 if toplam_islem > 0 else 0

print("=== YÖNETİM PANELİ ANA METRİKLERİ ===")
print(f"Toplam İşlem Adedi: {toplam_islem:,}")
print(f"Toplam İşlem Hacmi: {toplam_hacim:,.2f} TL")
print(f"Tespit Edilen Anomali Sayısı: {anomali_sayisi}")
print(f"Sistem Geneli Anomali Oranı: %{anomali_orani:.2f}")

In [ ]:
# Şehir bazlı gruplama (Biz transactions tablosundaki merchant_type veya atm_id üzerinden doğrudan KPI üretiyoruz)
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(12, 5))
islem_tipleri = df.groupby('transaction_type')['amount'].agg(['count', 'sum']).reset_index()

# Sol Grafik: İşlem Adetleri | Sağ Grafik: İşlem Hacimleri
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=islem_tipleri, x='transaction_type', y='count', ax=axes[0], hue='transaction_type', palette='Blues_r', legend=False)
axes[0].set_title('İşlem Tiplerine Göre Adet Dağılımı')
axes[0].set_ylabel('İşlem Sayısı')

sns.barplot(data=islem_tipleri, x='transaction_type', y='sum', ax=axes[1], hue='transaction_type', palette='Greens_r', legend=False)
axes[1].set_title('İşlem Tiplerine Göre Toplam Hacim (TL)')
axes[1].set_ylabel('Toplam Tutar (TL)')

plt.tight_layout()
# Grafiği raporlar klasörüne kaydeder
plt.savefig('../reports/kpi_dashboard_output.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os
import pandas as pd

print("=== KATMAN 2 ETL VERİ DOĞRULAMA TESTİ ===")

kpi_path = os.path.join('..', 'etl', 'load_targets', 'daily_kpi_report.csv')
anomaly_path = os.path.join('..', 'etl', 'load_targets', 'anomaly_summary_report.csv')

missing = []
if not os.path.exists(kpi_path):
    missing.append(kpi_path)
if not os.path.exists(anomaly_path):
    missing.append(anomaly_path)

if missing:
    print("Aşağıdaki ETL çıkış dosyaları bulunamadı:")
    for p in missing:
        print(" -", p)
    print()
    print("Lütfen önce ETL pipeline'ını çalıştırın (etl/extract.py) veya dosyaları etl/load_targets konumuna koyun.")
else:
    kpi_df = pd.read_csv(kpi_path)
    anomaly_df = pd.read_csv(anomaly_path)

    print(f"\n1. Daily KPI Raporu Satır Sayısı: {len(kpi_df)}")
    print(f"   Boş (Null) Değer Sayısı: {kpi_df.isnull().sum().sum()}")
    print("   Sütunlar:", list(kpi_df.columns))

    print(f"\n2. Anomali Özet Raporu Satır Sayısı: {len(anomaly_df)}")
    print(f"   Boş (Null) Değer Sayısı: {anomaly_df.isnull().sum().sum()}")
    if 'anomali_durumu' in anomaly_df.columns:
        print("\n   Anomali Dağılımı:\n", anomaly_df['anomali_durumu'].value_counts())
    else:
        print("\n   'anomali_durumu' sütunu bulunamadı. Sütun isimlerini kontrol edin:", list(anomaly_df.columns))

    print("\n=== DOĞRULAMA BAŞARILI! POWER BI İÇİN HAZIR ===")
